# MedGemma Spatial Transcriptomics — Benchmarking & Robustness Analysis

This notebook evaluates the pipeline across:
1. **Robustness**: multiple tissue types (breast, colon HD, prostate HD) with quality metrics
2. **Loki integration**: OmiCLIP embeddings vs z-score annotation comparison
3. **Report quality**: 5-metric benchmark comparing base vs LoRA MedGemma outputs

All data referenced is within the project directory.

In [ ]:
import sys
import os
import json
import time
import warnings
import tarfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/Users/sriharshameghadri/randomAIProjects/kaggle/medGemma')
sys.path.insert(0, str(PROJECT_ROOT))
# Also insert Loki src for import
sys.path.insert(0, str(PROJECT_ROOT / 'Loki' / 'src'))

sc.settings.verbosity = 1
print(f'scanpy {sc.__version__}')
print(f'Project root: {PROJECT_ROOT}')

## 1. Data Inventory

In [ ]:
SAMPLES = {
    'breast_10x': {
        'path': PROJECT_ROOT / 'outputs' / 'annotated_visium.h5ad',
        'tissue': 'Breast Cancer',
        'platform': 'Visium 10x',
        'status': 'annotated',
    },
    'colon_hd': {
        'path': PROJECT_ROOT / 'outputs' / 'test_full_markers_20260202_063705' /
                'e2735493904baddea063a55d5e676d24' / 'extracted' / 'square_008um.h5ad',
        'tissue': 'Colon Cancer',
        'platform': 'Visium HD 8µm',
        'status': 'raw',
    },
    'prostate_hd': {
        'path': PROJECT_ROOT / 'data' / 'raw' / 'prostate' /
                'Visium_HD_Human_Prostate_Cancer_FFPE_binned_outputs.tar.gz',
        'tissue': 'Prostate Cancer',
        'platform': 'Visium HD 8µm',
        'status': 'archived',
    },
}

for name, info in SAMPLES.items():
    exists = info['path'].exists()
    size_mb = info['path'].stat().st_size / 1e6 if exists else 0
    print(f"{name:20s} | {info['tissue']:20s} | {info['platform']:14s} | {'EXISTS' if exists else 'MISSING':8s} | {size_mb:.0f} MB")

## 2. Robustness Test — Breast Cancer (Baseline)

In [ ]:
from src.streamlit_adapter import annotate_spatial_regions, calculate_spatial_heterogeneity
from src.report_generation.prompt_builder import generate_medgemma_prompt, evaluate_report_quality


def run_pipeline_on_adata(adata, tissue_name, max_spots=5000):
    """Run full analysis pipeline on an AnnData object, with spot cap for memory."""
    t0 = time.time()
    result = {'tissue': tissue_name, 'n_spots_original': adata.n_obs}

    # Subsample large HD datasets
    if adata.n_obs > max_spots:
        sc.pp.subsample(adata, n_obs=max_spots, random_state=42)
        print(f'  Subsampled to {adata.n_obs} spots')

    result['n_spots_used'] = adata.n_obs
    result['n_genes'] = adata.n_vars

    adata_out = adata
    annot_metrics = {}
    spatial_metrics = {}

    try:
        adata_out, annot_metrics = annotate_spatial_regions(
            adata, resolution=0.5, use_markers=True, tissue=tissue_name
        )
        result['annotation_ok'] = True
        result['n_clusters'] = int(adata_out.obs['leiden'].nunique()) if 'leiden' in adata_out.obs else 0
    except Exception as e:
        result['annotation_ok'] = False
        result['annotation_error'] = str(e)
        result['n_clusters'] = 0
        print(f'  Annotation failed: {e}')

    try:
        spatial_metrics = calculate_spatial_heterogeneity(adata_out)
        result['features_ok'] = True
    except Exception as e:
        result['features_ok'] = False
        result['features_error'] = str(e)
        print(f'  Spatial metrics failed: {e}')

    # Assemble features dict (same structure as streamlit app)
    features = {
        'annotation': annot_metrics,
        'spatial_heterogeneity': spatial_metrics,
        'uncertainty': {
            'mean_prediction_entropy': float(adata_out.obs['prediction_entropy'].mean())
            if 'prediction_entropy' in adata_out.obs.columns else 1.0
        },
    }

    result['n_cell_types'] = len(
        (annot_metrics.get('cell_type_counts') or annot_metrics.get('cluster_distribution') or {})
    )
    result['morans_i'] = (
        spatial_metrics.get('morans_i', {}).get('mean', 0)
        if isinstance(spatial_metrics.get('morans_i'), dict)
        else spatial_metrics.get('morans_i_mean', 0)
    )
    result['mean_entropy'] = features['uncertainty']['mean_prediction_entropy']

    try:
        prompt = generate_medgemma_prompt(features, moa_focus=True)
        result['prompt_ok'] = True
        result['prompt_length'] = len(prompt)
    except Exception as e:
        result['prompt_ok'] = False
        print(f'  Prompt generation failed: {e}')

    result['elapsed_sec'] = round(time.time() - t0, 1)
    return result, adata_out, features

In [ ]:
print('Loading breast cancer sample...')
adata_breast = sc.read_h5ad(SAMPLES['breast_10x']['path'])
print(f'  {adata_breast.n_obs} spots, {adata_breast.n_vars} genes')

res_breast, adata_breast_out, features_breast = run_pipeline_on_adata(
    adata_breast.copy(), 'Breast Cancer'
)
print(f"\nBreast results:")
for k, v in res_breast.items():
    if not isinstance(v, dict):
        print(f"  {k}: {v}")

## 3. Robustness Test — Colon Cancer HD

In [ ]:
colon_path = SAMPLES['colon_hd']['path']

if colon_path.exists():
    print('Loading colon HD sample (516K spots — will subsample to 5000)...')
    adata_colon = sc.read_h5ad(colon_path)
    print(f'  {adata_colon.n_obs} spots, {adata_colon.n_vars} genes')

    res_colon, adata_colon_out, features_colon = run_pipeline_on_adata(
        adata_colon, 'Colon Cancer', max_spots=5000
    )
    print(f"\nColon results:")
    for k, v in res_colon.items():
        if not isinstance(v, dict):
            print(f"  {k}: {v}")
else:
    print('Colon h5ad not found, skipping.')
    res_colon = {'tissue': 'Colon Cancer', 'annotation_ok': False, 'features_ok': False}
    features_colon = {}

## 4. Robustness Test — Prostate Cancer HD (from archive)

In [ ]:
prostate_tar = SAMPLES['prostate_hd']['path']
res_prostate = None
features_prostate = {}

if prostate_tar.exists():
    print('Extracting prostate HD binned_outputs (3.2 GB archive)...')
    with tempfile.TemporaryDirectory() as tmpdir:
        with tarfile.open(prostate_tar, 'r:gz') as tar:
            # Only extract the 016um bin to save disk space
            members = [m for m in tar.getmembers()
                       if 'square_016um' in m.name and
                       ('filtered_feature_bc_matrix.h5' in m.name or
                        'spatial' in m.name)]
            tar.extractall(tmpdir, members=members)

        # Find the h5 file
        h5_files = list(Path(tmpdir).rglob('filtered_feature_bc_matrix.h5'))
        spatial_dirs = list(Path(tmpdir).rglob('spatial'))

        if h5_files:
            h5_path = h5_files[0]
            spatial_dir = spatial_dirs[0] if spatial_dirs else None
            print(f'  Found: {h5_path}')

            adata_prostate = sc.read_10x_h5(h5_path)
            if spatial_dir:
                adata_prostate.uns['spatial'] = {}
            print(f'  {adata_prostate.n_obs} spots, {adata_prostate.n_vars} genes')

            res_prostate, _, features_prostate = run_pipeline_on_adata(
                adata_prostate, 'Prostate Cancer', max_spots=5000
            )
            print(f"\nProstate results:")
            for k, v in res_prostate.items():
                if not isinstance(v, dict):
                    print(f"  {k}: {v}")
        else:
            print('  No h5 file found in 016um bin.')
            res_prostate = {'tissue': 'Prostate Cancer', 'annotation_ok': False}
else:
    print(f'Prostate archive not found at {prostate_tar}')
    res_prostate = {'tissue': 'Prostate Cancer', 'annotation_ok': False}

## 5. Robustness Summary Table

In [ ]:
all_results = [r for r in [res_breast, res_colon, res_prostate] if r is not None]

summary_rows = []
for r in all_results:
    summary_rows.append({
        'Tissue': r['tissue'],
        'Spots Used': r.get('n_spots_used', r.get('n_spots_original', '?')),
        'Genes': r.get('n_genes', '?'),
        'Clusters': r.get('n_clusters', 'N/A'),
        'Cell Types': r.get('n_cell_types', 'N/A'),
        "Moran's I": f"{r.get('morans_i', 0):.3f}",
        'Entropy': f"{r.get('mean_entropy', 0):.3f}",
        'Annotation OK': '✓' if r.get('annotation_ok') else '✗',
        'Features OK': '✓' if r.get('features_ok') else '✗',
        'Time (s)': r.get('elapsed_sec', '?'),
    })

df_summary = pd.DataFrame(summary_rows)
print('\n=== ROBUSTNESS SUMMARY ===')
print(df_summary.to_string(index=False))

# Save
os.makedirs(PROJECT_ROOT / 'outputs', exist_ok=True)
df_summary.to_csv(PROJECT_ROOT / 'outputs' / 'robustness_summary.csv', index=False)
print('\nSaved: outputs/robustness_summary.csv')

## 6. Visualization — Cluster Composition Across Tissues

In [ ]:
def get_cell_type_df(features, tissue_name):
    """Extract cell type proportions from features dict."""
    ct = (features.get('annotation', {}).get('cell_type_counts') or {})
    if not ct:
        return pd.DataFrame()
    total = sum(ct.values())
    return pd.DataFrame({
        'cell_type': list(ct.keys()),
        'proportion': [v / total for v in ct.values()],
        'tissue': tissue_name,
    })

frames = []
for res, feats in [(res_breast, features_breast), (res_colon, features_colon), (res_prostate, features_prostate)]:
    if res and feats:
        df = get_cell_type_df(feats, res['tissue'])
        if not df.empty:
            frames.append(df)

if frames:
    df_all = pd.concat(frames, ignore_index=True)

    tissues = df_all['tissue'].unique()
    fig, axes = plt.subplots(1, len(tissues), figsize=(6 * len(tissues), 5))
    if len(tissues) == 1:
        axes = [axes]

    colors = plt.cm.Set3.colors
    for ax, tissue in zip(axes, tissues):
        sub = df_all[df_all['tissue'] == tissue].sort_values('proportion', ascending=True)
        ax.barh(sub['cell_type'], sub['proportion'],
                color=[colors[i % len(colors)] for i in range(len(sub))])
        ax.set_title(tissue, fontsize=13, fontweight='bold')
        ax.set_xlabel('Proportion')
        ax.set_xlim(0, max(sub['proportion'].max() * 1.1, 0.1))
        ax.grid(axis='x', alpha=0.3)

    plt.suptitle('Cell Type Composition Across Tissues', fontsize=15, y=1.02)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'robustness_cell_types.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/robustness_cell_types.png')
else:
    print('No cell type data to plot.')

## 7. Loki Integration — OmiCLIP Embeddings

In [ ]:
LOKI_CHECKPOINT = PROJECT_ROOT / 'data' / 'checkpoint.pt'
HOUSEKEEPING_GENES = PROJECT_ROOT / 'data' / 'housekeeping_genes.csv'

print(f'Loki checkpoint: {LOKI_CHECKPOINT.exists()} ({LOKI_CHECKPOINT.stat().st_size / 1e9:.1f} GB)' if LOKI_CHECKPOINT.exists() else 'Loki checkpoint: MISSING')
print(f'Housekeeping genes: {HOUSEKEEPING_GENES.exists()}')

LOKI_AVAILABLE = LOKI_CHECKPOINT.exists() and HOUSEKEEPING_GENES.exists()

In [ ]:
loki_results = {}

if LOKI_AVAILABLE:
    import torch
    import open_clip.factory as _ocf

    # PyTorch 2.6 compatibility: Loki checkpoint has numpy globals
    _orig_load = _ocf.load_state_dict
    _ocf.load_state_dict = lambda p, device=None, weights_only=True: _orig_load(p, device=device, weights_only=False)

    import loki.utils
    import loki.preprocess
    import scipy.sparse as sp

    print('Loading Loki OmiCLIP model (this takes ~60-90s on CPU)...')
    t0 = time.time()
    model, preprocess_fn, tokenizer = loki.utils.load_model(str(LOKI_CHECKPOINT), 'cpu')
    model.eval()
    print(f'  Model loaded in {time.time() - t0:.1f}s')

    house_keeping_genes = pd.read_csv(HOUSEKEEPING_GENES, index_col=0)

    def run_loki_on_sample(adata_sample, name, max_spots_loki=500):
        """Generate Loki text embeddings for a sample."""
        t0 = time.time()

        # Subsample for speed (Loki is slow on CPU)
        if adata_sample.n_obs > max_spots_loki:
            sc.pp.subsample(adata_sample, n_obs=max_spots_loki, random_state=42)

        try:
            gene_df = loki.preprocess.generate_gene_df(
                adata_sample, house_keeping_genes, todense=sp.issparse(adata_sample.X)
            )
            embeddings = loki.utils.encode_text_df(
                model, tokenizer, gene_df, 'label', 'cpu'
            )
            emb_np = embeddings.cpu().numpy()

            # Cluster embeddings
            from sklearn.cluster import KMeans
            n_clusters = min(5, adata_sample.n_obs // 10)
            kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
            loki_labels = kmeans.fit_predict(emb_np)

            # Embedding statistics
            norms = np.linalg.norm(emb_np, axis=1)

            result = {
                'name': name,
                'status': 'ok',
                'n_spots': adata_sample.n_obs,
                'embedding_dim': emb_np.shape[1],
                'n_loki_clusters': n_clusters,
                'cluster_counts': dict(pd.Series(loki_labels).value_counts().to_dict()),
                'mean_norm': float(norms.mean()),
                'std_norm': float(norms.std()),
                'elapsed_sec': round(time.time() - t0, 1),
                'embeddings': emb_np,
                'loki_labels': loki_labels,
            }
            print(f'  {name}: {emb_np.shape} embeddings in {result["elapsed_sec"]}s')
            return result

        except Exception as e:
            print(f'  {name} FAILED: {e}')
            return {'name': name, 'status': 'failed', 'error': str(e)}

    print('\nRunning Loki on breast cancer sample...')
    loki_results['breast'] = run_loki_on_sample(adata_breast.copy(), 'Breast Cancer')

    if SAMPLES['colon_hd']['path'].exists():
        print('\nRunning Loki on colon HD sample...')
        adata_colon_loki = sc.read_h5ad(SAMPLES['colon_hd']['path'])
        loki_results['colon'] = run_loki_on_sample(adata_colon_loki, 'Colon Cancer')

else:
    print('Loki not available — checkpoint or housekeeping genes missing')
    print('  Expected: data/checkpoint.pt + data/housekeeping_genes.csv')
    loki_results = {}

## 8. Loki UMAP Visualization

In [ ]:
ok_loki = {k: v for k, v in loki_results.items() if v.get('status') == 'ok'}

if ok_loki:
    from umap import UMAP

    n_panels = len(ok_loki)
    fig, axes = plt.subplots(1, n_panels, figsize=(7 * n_panels, 6))
    if n_panels == 1:
        axes = [axes]

    for ax, (key, res) in zip(axes, ok_loki.items()):
        reducer = UMAP(n_components=2, random_state=42)
        coords_2d = reducer.fit_transform(res['embeddings'])
        scatter = ax.scatter(
            coords_2d[:, 0], coords_2d[:, 1],
            c=res['loki_labels'], cmap='tab10', s=8, alpha=0.7
        )
        plt.colorbar(scatter, ax=ax, label='Loki Cluster')
        ax.set_title(f"{res['name']}\nLoki UMAP ({res['n_spots']} spots, dim={res['embedding_dim']})",
                     fontsize=12)
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
        ax.grid(alpha=0.2)

    plt.suptitle('Loki OmiCLIP Embedding Space', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'outputs' / 'loki_umap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/loki_umap.png')
else:
    print('No Loki embeddings available for UMAP.')

## 9. Report Quality Benchmark — 5-Metric Evaluation

In [ ]:
from src.report_generation.prompt_builder import (
    generate_medgemma_prompt,
    evaluate_report_quality,
    create_anti_parroting_prompt,
)


def score_report(report_text, features):
    """Run 5-metric benchmark on a report."""
    quality = evaluate_report_quality(report_text, features)

    moa_terms = ['pathway', 'signaling', 'mechanism', 'inhibit', 'activate',
                 'receptor', 'ligand', 'target', 'checkpoint', 'immunotherapy',
                 'resistance', 'proliferation', 'apoptosis', 'immunosuppression']
    research_terms = ['suggests', 'consistent with', 'indicative of',
                      'associated with', 'may reflect', 'warrants']
    mitigation_terms = ['caution', 'limitations', 'uncertainty', 'confidence',
                        'should be', 'further validation', 'consider']

    report_lower = report_text.lower()
    moa_hits = sum(1 for t in moa_terms if t in report_lower)
    research_hits = sum(1 for t in research_terms if t in report_lower)
    mitigation_hits = sum(1 for t in mitigation_terms if t in report_lower)

    metrics = {
        'word_count': quality.get('word_count', 0),
        'moa_terms': moa_hits,
        'research_linkage': research_hits,
        'mitigation_terms': mitigation_hits,
        'parroting_risk': quality.get('parroting_risk', 'UNKNOWN'),
        'has_interpretation': quality.get('has_interpretation', False),
    }

    # GO / NO-GO decisions per metric
    go_nogo = {
        'word_count': 'GO' if 250 <= metrics['word_count'] <= 500 else 'NO-GO',
        'moa_terms': 'GO' if metrics['moa_terms'] >= 3 else 'NO-GO',
        'research_linkage': 'GO' if metrics['research_linkage'] >= 1 else 'NO-GO',
        'mitigation_terms': 'GO' if metrics['mitigation_terms'] >= 2 else 'NO-GO',
        'parroting_risk': 'GO' if metrics['parroting_risk'] in ('LOW', 'MODERATE') else 'NO-GO',
    }

    metrics['go_nogo'] = go_nogo
    metrics['n_go'] = sum(1 for v in go_nogo.values() if v == 'GO')
    metrics['overall'] = 'GO' if metrics['n_go'] >= 4 else 'NO-GO'
    return metrics


print('Report quality benchmark ready.')

In [ ]:
# Synthetic test reports to benchmark — simulate base vs fine-tuned outputs
# In production, replace these with actual MedGemma / LoRA model outputs

SYNTHETIC_BASE = """The spatial transcriptomics analysis reveals a complex tumor microenvironment.
T cells account for 25% of spots, Macrophages 18%, Epithelial cells 35%, and Fibroblasts 12%.
Moran's I = 0.45 indicating moderate spatial autocorrelation.
There are 3 spatially enriched co-localization pairs.
The entropy is 1.2 indicating moderate uncertainty.
Cluster 0 has 300 spots, Cluster 1 has 200 spots, Cluster 2 has 150 spots."""

SYNTHETIC_LORA = """The spatial architecture of this tumor microenvironment suggests active immune
infiltration consistent with a T cell-inflamed phenotype. The predominant epithelial population
(35%) shows spatial clustering patterns indicative of organized tumor nests, while the significant
T cell infiltration (25%) co-localizing with macrophages (18%) warrants consideration of
checkpoint immunotherapy sensitivity. The PD-1/PD-L1 axis and LAG-3 signaling pathway may be
active given these immune proportions. Spatial autocorrelation (Moran's I=0.45) suggests
non-random cellular organization, potentially reflecting ongoing immune-epithelial crosstalk
via IFN-γ and TGF-β signaling mechanisms. The stromal fibroblast component (12%) may activate
pro-tumorigenic pathways including FAK and integrin signaling. Caution: the moderate prediction
entropy indicates uncertainty in cell type assignments; further validation with IHC for CD8,
CD163, and CK markers should be considered before clinical decision-making."""

print('Scoring synthetic base model report...')
score_base = score_report(SYNTHETIC_BASE, features_breast)
print('\nScoring synthetic LoRA fine-tuned report...')
score_lora = score_report(SYNTHETIC_LORA, features_breast)

benchmark_df = pd.DataFrame({
    'Metric': ['Word Count (250-500)', 'MoA Terms (≥3)', 'Research Linkage (≥1)',
               'Mitigation Terms (≥2)', 'Anti-Parroting (LOW/MOD)', 'OVERALL', 'n_go/5'],
    'Base Model': [
        f"{score_base['word_count']} ({score_base['go_nogo']['word_count']})",
        f"{score_base['moa_terms']} ({score_base['go_nogo']['moa_terms']})",
        f"{score_base['research_linkage']} ({score_base['go_nogo']['research_linkage']})",
        f"{score_base['mitigation_terms']} ({score_base['go_nogo']['mitigation_terms']})",
        f"{score_base['parroting_risk']} ({score_base['go_nogo']['parroting_risk']})",
        score_base['overall'],
        f"{score_base['n_go']}/5",
    ],
    'LoRA Fine-tuned': [
        f"{score_lora['word_count']} ({score_lora['go_nogo']['word_count']})",
        f"{score_lora['moa_terms']} ({score_lora['go_nogo']['moa_terms']})",
        f"{score_lora['research_linkage']} ({score_lora['go_nogo']['research_linkage']})",
        f"{score_lora['mitigation_terms']} ({score_lora['go_nogo']['mitigation_terms']})",
        f"{score_lora['parroting_risk']} ({score_lora['go_nogo']['parroting_risk']})",
        score_lora['overall'],
        f"{score_lora['n_go']}/5",
    ],
})

print('\n=== 5-METRIC BENCHMARK ===')
print(benchmark_df.to_string(index=False))

benchmark_df.to_csv(PROJECT_ROOT / 'outputs' / 'report_benchmark.csv', index=False)
print('\nSaved: outputs/report_benchmark.csv')

## 10. Benchmark with Real LoRA Adapter (if available)

In [ ]:
# Check if LoRA adapter is available locally
ADAPTER_PATH = PROJECT_ROOT / 'outputs' / 'lora_adapter'

if ADAPTER_PATH.exists():
    print(f'LoRA adapter found at {ADAPTER_PATH}')
    print('Loading MedGemma base + LoRA adapter for real benchmark...')
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        from peft import PeftModel

        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                        bnb_4bit_compute_dtype=torch.float16)
        MODEL_ID = 'google/medgemma-4b-it'
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=bnb_config, device_map='auto'
        )
        lora_model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))

        def generate_report(model, tokenizer, features, max_new_tokens=400):
            prompt = create_anti_parroting_prompt(features)
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=max_new_tokens,
                    temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id
                )
            return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

        prompt = create_anti_parroting_prompt(features_breast)
        print('\nGenerating base model report...')
        report_base = generate_report(base_model, tokenizer, features_breast)
        print('Generating LoRA report...')
        report_lora = generate_report(lora_model, tokenizer, features_breast)

        score_real_base = score_report(report_base, features_breast)
        score_real_lora = score_report(report_lora, features_breast)

        print(f'\nBase: {score_real_base["n_go"]}/5 GO  |  LoRA: {score_real_lora["n_go"]}/5 GO')

        with open(PROJECT_ROOT / 'outputs' / 'lora_benchmark_results.json', 'w') as f:
            json.dump({'base': score_real_base, 'lora': score_real_lora,
                       'base_report': report_base[:500], 'lora_report': report_lora[:500]}, f, indent=2)
        print('Saved: outputs/lora_benchmark_results.json')

    except Exception as e:
        print(f'LoRA benchmark failed: {e}')
        print('Run LoRA training first (see notebooks/train_lora.ipynb on Kaggle T4)')
else:
    print(f'LoRA adapter not found at {ADAPTER_PATH}')
    print('Using synthetic benchmark scores from Cell 9.')
    print('To run real benchmark: complete LoRA training on Kaggle, download adapter.')

## 11. Loki vs Z-Score Annotation Comparison

In [ ]:
if ok_loki and features_breast:
    # Compare cluster structure: Loki clusters vs z-score cell type assignments
    loki_br = ok_loki.get('breast')
    zscore_ct = features_breast.get('annotation', {}).get('cell_type_counts') or {}

    if loki_br and zscore_ct:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Z-score annotation
        labels = list(zscore_ct.keys())
        values = list(zscore_ct.values())
        total = sum(values)
        axes[0].pie(
            values,
            labels=[f"{l}\n({v/total:.0%})" for l, v in zip(labels, values)],
            colors=plt.cm.Set3.colors[:len(labels)],
            startangle=90
        )
        axes[0].set_title('Z-Score Annotation\n(CellTypist + compartment markers)', fontsize=12)

        # Loki cluster distribution
        loki_counts = loki_br['cluster_counts']
        axes[1].bar(
            [f'Cluster {k}' for k in sorted(loki_counts.keys())],
            [loki_counts[k] for k in sorted(loki_counts.keys())],
            color=plt.cm.tab10.colors[:len(loki_counts)]
        )
        axes[1].set_title('Loki OmiCLIP Clusters\n(Unsupervised gene embedding KMeans)', fontsize=12)
        axes[1].set_ylabel('Number of spots')
        axes[1].grid(axis='y', alpha=0.3)

        plt.suptitle('Breast Cancer: Annotation Strategy Comparison', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(PROJECT_ROOT / 'outputs' / 'annotation_comparison.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: outputs/annotation_comparison.png')
    else:
        print('Missing data for annotation comparison.')
else:
    print('Loki not run — skipping annotation comparison.')

## 12. Final Results Summary & Saving

In [ ]:
final = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'robustness': {
        'breast': {k: v for k, v in res_breast.items() if not isinstance(v, dict)},
        'colon': {k: v for k, v in res_colon.items() if not isinstance(v, dict)},
    },
    'loki': {
        k: {kk: vv for kk, vv in v.items() if kk not in ('embeddings', 'loki_labels')}
        for k, v in loki_results.items()
    },
    'benchmark_synthetic': {
        'base_n_go': score_base['n_go'],
        'lora_n_go': score_lora['n_go'],
        'base_overall': score_base['overall'],
        'lora_overall': score_lora['overall'],
    },
}

if res_prostate:
    final['robustness']['prostate'] = {k: v for k, v in res_prostate.items() if not isinstance(v, dict)}

out_path = PROJECT_ROOT / 'outputs' / 'benchmarking_results.json'
with open(out_path, 'w') as f:
    json.dump(final, f, indent=2, default=str)

print('=' * 60)
print('BENCHMARKING COMPLETE')
print('=' * 60)
print(f'Results saved: {out_path}')
print()
print('Outputs generated:')
for fname in ['robustness_summary.csv', 'robustness_cell_types.png',
              'loki_umap.png', 'annotation_comparison.png',
              'report_benchmark.csv', 'benchmarking_results.json']:
    p = PROJECT_ROOT / 'outputs' / fname
    print(f'  {"OK" if p.exists() else "MISSING"}: {fname}')